# LISA Dataset

### Do the same thing above for the LISA dataset off of Roboflow

In [19]:
import os
import csv
import yaml
import math
import random
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Image as IPImage
import ipywidgets as widgets

from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
%matplotlib inline

In [20]:
LISA_ROOT    = Path("dataset/lisa")      # change if needed
OUTPUT_DIR   = Path("lisa_contact_sheets")
CSV_PATH     = "lisa_review.csv"

SPLITS       = ["train", "val", "test"]
N_SAMPLES    = 12
COLS         = 4
CROP_PAD     = 0.35
DPI          = 80

OUTPUT_DIR.mkdir(exist_ok=True)

In [21]:
def load_yaml(lisa_root):
    yaml_path = lisa_root / "data.yaml"
    with open(yaml_path) as f:
        data = yaml.safe_load(f)
    # Roboflow puts class names under 'names' as a list
    names = data["names"]
    # Normalise: could be a list or a {id: name} dict
    if isinstance(names, dict):
        id_to_name = {int(k): v for k, v in names.items()}
    else:
        id_to_name = {i: n for i, n in enumerate(names)}
    return id_to_name

id_to_name = load_yaml(LISA_ROOT)
name_to_id = {v: k for k, v in id_to_name.items()}
print(f"{len(id_to_name)} classes found:")
for cid, name in sorted(id_to_name.items()):
    print(f"  {cid:3d}  {name}")

60 classes found:
    0  addedLane
    1  curveLeft
    2  curveRight
    3  dip
    4  doNotEnter
    5  doNotPass
    6  go
    7  go forward
    8  go forward traffic light
    9  go left
   10  go left traffic light
   11  go traffic light
   12  intersection
   13  keepRight
   14  laneEnds
   15  merge
   16  noLeftTurn
   17  noRightTurn
   18  pedestrianCrossing
   19  rampSpeedAdvisory20
   20  rampSpeedAdvisory35
   21  rampSpeedAdvisory40
   22  rampSpeedAdvisory45
   23  rampSpeedAdvisory50
   24  rampSpeedAdvisoryUrdbl
   25  rightLaneMustTurn
   26  roundabout
   27  school
   28  schoolSpeedLimit25
   29  signalAhead
   30  slow
   31  speedLimit15
   32  speedLimit25
   33  speedLimit30
   34  speedLimit35
   35  speedLimit40
   36  speedLimit45
   37  speedLimit50
   38  speedLimit55
   39  speedLimit65
   40  speedLimitUrdbl
   41  stop
   42  stop left
   43  stop left traffic light
   44  stop traffic light
   45  stopAhead
   46  thruMergeLeft
   47  thruMergeRight

In [22]:
def collect_lisa_crops(lisa_root, splits, id_to_name):
    """
    Returns:
        counts  : {class_name: int}
        samples : {class_name: [{image_path, bbox_yolo}, ...]}
    """
    counts  = defaultdict(int)
    samples = defaultdict(list)

    for split in splits:
        labels_dir = lisa_root / split / "labels"
        images_dir = lisa_root / split / "images"
        if not labels_dir.exists():
            print(f"[warn] not found: {labels_dir}")
            continue

        for label_file in labels_dir.glob("*.txt"):
            # Find matching image (try common extensions)
            img_path = None
            for ext in [".jpg", ".jpeg", ".png"]:
                candidate = images_dir / (label_file.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break

            with open(label_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    class_id = int(parts[0])
                    name     = id_to_name.get(class_id, f"unknown_{class_id}")
                    counts[name] += 1

                    if img_path and len(samples[name]) < 50:  # cap stored samples
                        samples[name].append({
                            "image_path": img_path,
                            "bbox_yolo":  [float(x) for x in parts[1:5]],
                        })

    return counts, samples

counts, samples = collect_lisa_crops(LISA_ROOT, SPLITS, id_to_name)
print(f"Collected annotation counts for {len(counts)} classes.")

Collected annotation counts for 60 classes.


In [23]:
def print_count_summary(counts):
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    total = sum(counts.values())
    arr   = np.array(list(counts.values()))

    print(f"{'Class':<55} {'Count':>7}")
    print("─" * 64)
    for name, count in sorted_counts:
        bar = "█" * min(30, int(count / max(arr) * 30))
        print(f"  {name:<53} {count:>6,}  {bar}")
    print("─" * 64)
    print(f"  {'Total annotations':<53} {total:>6,}")
    print(f"  {'Mean per class':<53} {arr.mean():>6.1f}")
    print(f"  {'Median per class':<53} {np.median(arr):>6.1f}")
    print(f"  {'Min':<53} {arr.min():>6,}")
    print(f"  {'Max':<53} {arr.max():>6,}")
    print(f"  {'Classes < 100':<53} {(arr < 100).sum():>6}")
    print(f"  {'Classes < 500':<53} {(arr < 500).sum():>6}")

print_count_summary(counts)

Class                                                     Count
────────────────────────────────────────────────────────────────
  stop                                                   3,504  ██████████████████████████████
  pedestrianCrossing                                     1,440  ████████████
  signalAhead                                            1,247  ██████████
  go                                                     1,079  █████████
  go traffic light                                       1,069  █████████
  stop traffic light                                     1,033  ████████
  speedLimit35                                             722  ██████
  speedLimit25                                             454  ███
  keepRight                                                444  ███
  stop left traffic light                                  419  ███
  stop left                                                419  ███
  addedLane                                                4

In [24]:
def yolo_to_pixel(bbox_yolo, img_w, img_h, pad):
    cx, cy, bw, bh = bbox_yolo
    xmin = (cx - bw / 2) * img_w
    ymin = (cy - bh / 2) * img_h
    xmax = (cx + bw / 2) * img_w
    ymax = (cy + bh / 2) * img_h
    pw   = (xmax - xmin) * pad
    ph   = (ymax - ymin) * pad
    x1   = max(0,      xmin - pw)
    y1   = max(0,      ymin - ph)
    x2   = min(img_w,  xmax + pw)
    y2   = min(img_h,  ymax + ph)
    return x1, y1, x2, y2


def save_contact_sheet_lisa(class_name, class_samples):
    picks = class_samples[:N_SAMPLES]
    rows  = math.ceil(len(picks) / COLS)

    fig, axes = plt.subplots(rows, COLS,
                             figsize=(COLS * 3, rows * 3),
                             squeeze=False)
    fig.suptitle(class_name, fontsize=9, fontweight="bold")

    shown = 0
    for s in picks:
        try:
            img  = Image.open(s["image_path"]).convert("RGB")
            x1, y1, x2, y2 = yolo_to_pixel(s["bbox_yolo"], img.width, img.height, CROP_PAD)
            crop = img.crop((x1, y1, x2, y2))
            r, c = divmod(shown, COLS)
            axes[r][c].imshow(crop)
            axes[r][c].axis("off")
            shown += 1
        except Exception as e:
            print(f"  [warn] {s['image_path'].name}: {e}")

    for idx in range(shown, rows * COLS):
        r, c = divmod(idx, COLS)
        axes[r][c].axis("off")

    safe   = class_name.replace("/", "_").replace("\\", "_")
    out    = OUTPUT_DIR / f"{safe}.png"
    plt.savefig(out, bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    return out


all_names = sorted(counts.keys())
total     = len(all_names)

for i, name in enumerate(all_names, 1):
    out = OUTPUT_DIR / f"{name.replace('/', '_')}.png"
    if out.exists():
        continue
    save_contact_sheet_lisa(name, samples[name])
    print(f"  [{i}/{total}] {name}")

print("All contact sheets saved.")

All contact sheets saved.


In [25]:
# ── Load existing decisions ───────────────────────────────────────────────────
decisions = {}  # lisa_name -> {keep, mapillary_label, notes}

if Path(CSV_PATH).exists():
    with open(CSV_PATH, newline="") as f:
        for row in csv.DictReader(f):
            decisions[row["lisa_label"]] = {
                "keep":            row["keep"],
                "mapillary_label": row["mapillary_label"],
                "notes":           row["notes"],
            }

def save_csv():
    with open(CSV_PATH, "w", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=["lisa_label", "sample_count", "keep",
                           "mapillary_label", "notes"])
        writer.writeheader()
        for name in sorted(counts.keys()):
            d = decisions.get(name, {"keep": "", "mapillary_label": "", "notes": ""})
            writer.writerow({
                "lisa_label":      name,
                "sample_count":    counts[name],
                "keep":            d["keep"],
                "mapillary_label": d["mapillary_label"],
                "notes":           d["notes"],
            })

unreviewed = [n for n in all_names if decisions.get(n, {}).get("keep", "") == ""]
start_idx  = all_names.index(unreviewed[0]) if unreviewed else 0
state      = {"idx": start_idx}

print(f"{len(all_names) - len(unreviewed)}/{len(all_names)} already reviewed. "
      f"Starting at index {start_idx}.")

60/60 already reviewed. Starting at index 0.


In [26]:
# ── Widget UI ─────────────────────────────────────────────────────────────────

img_out      = widgets.Output()
status_label = widgets.Label()

mapillary_box = widgets.Text(
    placeholder="Mapillary label (leave blank to keep LISA name)",
    layout=widgets.Layout(width="500px")
)
notes_box = widgets.Text(
    placeholder="optional notes",
    layout=widgets.Layout(width="500px")
)

def make_btn(label, color):
    return widgets.Button(description=label,
                          style={"button_color": color},
                          layout=widgets.Layout(width="130px"))

btn_keep  = make_btn("✅ Keep",   "#c8f7c5")
btn_skip  = make_btn("❌ Skip",   "#fdd")
btn_prev  = make_btn("⬅ Prev",   "#e0e0e0")
btn_jump  = make_btn("↩ Jump to…","#e0e0e0")
jump_input = widgets.IntText(value=1, layout=widgets.Layout(width="80px"))


def show_current():
    idx  = state["idx"]
    name = all_names[idx]
    d    = decisions.get(name, {})

    reviewed = len([n for n in all_names if decisions.get(n, {}).get("keep", "") != ""])
    status_label.value = (
        f"[{idx+1}/{len(all_names)}]  "
        f"{reviewed} reviewed, {len(unreviewed)} remaining  |  "
        f"{name}  ({counts[name]:,} annotations)"
    )

    # Pre-fill fields with any existing decision
    mapillary_box.value = d.get("mapillary_label", "")
    notes_box.value     = d.get("notes", "")

    sheet_path = OUTPUT_DIR / f"{name.replace('/', '_')}.png"
    with img_out:
        img_out.clear_output(wait=True)
        if sheet_path.exists():
            display(IPImage(filename=str(sheet_path), width=800))
        else:
            print(f"Contact sheet not found: {sheet_path}")


def record(keep_value):
    name = all_names[state["idx"]]
    mapped = mapillary_box.value.strip()
    decisions[name] = {
        "keep":            keep_value,
        "mapillary_label": mapped if mapped else name,
        "notes":           notes_box.value.strip(),
    }
    save_csv()
    if state["idx"] < len(all_names) - 1:
        state["idx"] += 1
    show_current()


btn_keep.on_click(lambda _: record("yes"))
btn_skip.on_click(lambda _: record("no"))

def go_prev(_):
    if state["idx"] > 0:
        state["idx"] -= 1
    show_current()

def go_jump(_):
    state["idx"] = max(0, min(len(all_names) - 1, jump_input.value - 1))
    show_current()

btn_prev.on_click(go_prev)
btn_jump.on_click(go_jump)

ui = widgets.VBox([
    status_label,
    img_out,
    widgets.HBox([btn_keep, btn_skip, btn_prev]),
    widgets.HBox([widgets.Label("Mapillary label:"), mapillary_box]),
    widgets.HBox([widgets.Label("Notes:         "), notes_box]),
    widgets.HBox([btn_jump, jump_input, widgets.Label("(1-indexed)")]),
])

show_current()
display(ui)

In [27]:
from collections import Counter

keep_classes = [
    (n, decisions[n]["mapillary_label"], counts[n])
    for n in all_names
    if decisions.get(n, {}).get("keep") == "yes"
]

decision_counts = Counter(d["keep"] for d in decisions.values())

print(f"Total classes  : {len(all_names)}")
print(f"Reviewed       : {sum(decision_counts.values())}")
print(f"  Keep         : {decision_counts.get('yes', 0)}")
print(f"  Skip         : {decision_counts.get('no', 0)}")
print(f"  Unreviewed   : {len(all_names) - sum(decision_counts.values())}")
print()
print(f"{'LISA label':<40} {'Mapillary mapping':<55} {'Count':>7}")
print("─" * 104)
for lisa_name, mapillary_name, count in sorted(keep_classes, key=lambda x: x[2], reverse=True):
    same = "" if lisa_name != mapillary_name else "  (unchanged)"
    print(f"  {lisa_name:<38} → {mapillary_name:<53} {count:>6,}{same}")

Total classes  : 60
Reviewed       : 60
  Keep         : 27
  Skip         : 33
  Unreviewed   : 0

LISA label                               Mapillary mapping                                         Count
────────────────────────────────────────────────────────────────────────────────────────────────────────
  pedestrianCrossing                     → warning--pedestrians-crossing--g4                      1,440
  signalAhead                            → warning--traffic-signals--g3                           1,247
  speedLimit35                           → regulatory--maximum-speed-limit-35--g2                   722
  speedLimit25                           → regulatory--maximum-speed-limit-25--g2                   454
  keepRight                              → complementary--keep-right--g1                            444
  addedLane                              → warning--added-lane-right--g1                            416
  merge                                  → warning--traffic-merges